# PIPELINE STREAMING SANS SPARK DECLARATIVE PIPELINE

Le but de ce projet est de s'obliger à utiliser spark structured streaming plutôt que spark declarative pipelines afin d'effectuer un traitement suivant l'architecture medallion en near real time sur des données de retail.
Les données utilisées ici proviennent du marketplace Databricks et il s'agit de données de retail. 
Les données se trouve dans un 3 volumes de fichiers json: **customers**, **orders**, **status**. La table **customer** répresente les infos des clients, qui change dans le temps donc sur cette table on implémentera un SCD de type 1 afin de récuperer les informations récentes des clients. La table **orders** représente l'historiques des commandes, on fera donc un chargement incrémental(upsert) pour le transfert des données de la couche bronze à la couche silver. Enfin pour la table **status** qui représente le statut des commandes, on implémetera aussi une logique de SCD étant donné que le statut des commandes évolue dans le temps.

In [0]:
import pyspark.sql.functions as F

## Exploration des differentes sources

In [0]:
%sql
USE CATALOG spark_struct_retail;

In [0]:
%sql
SELECT *
FROM json.`/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/stream_json`
LIMIT 10;

In [0]:
%sql
SELECT * 
FROM json. `/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json`;

In [0]:
%sql
SELECT * 
FROM json. `/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/stream_json`;

## Couche Bronze

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/stream_json",
    format => 'json',
    multiline => 'false'
)

In [0]:
query = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/spark_struct_retail/checkpoints/customers")
    .option("schemaEvolution", "rescue")
    .load("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/")
    .writeStream
    .option("checkpointLocation","/Volumes/spark_struct_retail/checkpoints/customers")
    .option("mergeSchema", True)
    .trigger(availableNow = True)
    .table("spark_struct_retail.bronze.customers_bronze")
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM spark_struct_retail.bronze.customers_bronze
LIMIT 10;

In [0]:
query = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/spark_struct_retail/checkpoints/orders")
    .option("schemaEvolution", "rescue")
    .load("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/")
    .writeStream
    .option("checkpointLocation", "/Volumes/spark_struct_retail/checkpoints/orders")
    .option("mergeSchema", True)
    .trigger(availableNow = True)
    .table("spark_struct_retail.bronze.orders_bronze")
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM spark_struct_retail.bronze.orders_bronze
LIMIT 500;

In [0]:
query = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/spark_struct_retail/checkpoints/status")
    .option("schemaEvolution", "rescue")
    .load("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/")
    .writeStream
    .option("checkpointLocation", "/Volumes/spark_struct_retail/checkpoints/status")
    .option("mergeSchema", True)
    .trigger(availableNow = True)
    .table("spark_struct_retail.bronze.status_bronze")
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM spark_struct_retail.bronze.status_bronze
LIMIT 100;

In [0]:
(
    spark
    .read
    .table("spark_struct_retail.bronze.status_bronze"
    )
    .select(["order_status"])
    .distinct()
    .show()
)

## Couche Silver

**customers table**

In [0]:
from pyspark.sql.window import Window
def upsert_batch(micro_batch, batch_id):

    window = (
        Window
        .partitionBy("customer_id")
        .orderBy(F.desc("timestamp"))
    )

    (
        micro_batch
        .withColumn(
            "customer_id",
            F.col("customer_id").cast("integer")
        )
        .withColumn(
            "timestamp",
            F.from_unixtime(
                F.col("timestamp").cast("long")
            ).cast("timestamp")
        )
        .withColumn(
            "rn",
            F.row_number().over(window)
        )
        .filter(F.col("rn") == 1)
        .filter(F.col("operation").isin("UPDATE", "INSERT", "NEW"))
        .createOrReplaceTempView("micro_batch_vw")
    )

    sql_query = """
        MERGE INTO spark_struct_retail.silver.customers_silver AS t
        USING micro_batch_vw AS s

        ON t.customer_id = s.customer_id

        WHEN MATCHED AND s.operation = 'UPDATE'
        THEN UPDATE SET *

        WHEN NOT MATCHED 
        THEN INSERT *
    """

    micro_batch.sparkSession.sql(sql_query)

In [0]:
%sql
CREATE OR REPLACE TABLE spark_struct_retail.silver.customers_silver(
    address STRING, 
    city STRING, 
    customer_id INT, 
    state STRING, 
    timestamp TIMESTAMP, 
    zip_code STRING
)

In [0]:
(spark.
 readStream
.table("spark_struct_retail.bronze.customers_bronze")
.writeStream
.foreachBatch(upsert_batch)
.option("checkpointLocation", "/Volumes/spark_struct_retail/checkpoints/customers_silver/")
.trigger(availableNow = True)
.start()
)

In [0]:
%sql
select * from spark_struct_retail.silver.customers_silver

**orders table**

In [0]:
df_deduped = (
    spark.readStream
    .table("spark_struct_retail.bronze.orders_bronze")
    .withColumn(
        "customer_id",
        F.col("customer_id").cast("integer")
    )
    .withColumn(
        "order_id",
        F.col("order_id").cast("integer")
    )
    .withColumn(
        "order_timestamp",
        F.from_unixtime(
            F.col("order_timestamp").cast("long")
        ).cast("timestamp")
    )
    .withWatermark("order_timestamp", "30 seconds")
    .dropDuplicates(["order_id"])
    .dropna(how="all")
    .withColumn(
        "year",
        F.year(F.col("order_timestamp"))
    )
    .withColumn(
        "month",
        F.month(F.col("order_timestamp"))
    )
    .withColumn(
        "weekday",
        F.weekday(F.col("order_timestamp"))
    )
)

In [0]:
def upsert_batch(microbatch, batch):

    microbatch.createOrReplaceTempView("microbatch_view")

    sql_query = """
    MERGE INTO spark_struct_retail.silver.orders_silver t
    USING microbatch_view s
    ON t.order_id = s.order_id
    WHEN NOT MATCHED THEN
    INSERT *
    """
    
    microbatch.sparkSession.sql(sql_query)

In [0]:
%sql
CREATE OR REPLACE TABLE spark_struct_retail.silver.orders_silver(
    customer_id INT, notifications STRING, order_id INT, order_timestamp TIMESTAMP, year INT, month INT, weekday INT
);

In [0]:
(
    df_deduped
    .writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", "/Volumes/spark_struct_retail/checkpoints/orders_silver/")
    .trigger(availableNow = True)
    .start()
)

In [0]:
%sql
SELECT * FROM spark_struct_retail.silver.orders_silver;

**status table**

In [0]:
%sql
SELECT * FROM spark_struct_retail.bronze.status_bronze
LIMIT 10;

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

def upsert_batch(microbatch, batch):

    window = Window.partitionBy("order_id").orderBy(F.desc("status_timestamp"))

    (
        microbatch
        .withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn")
        .withColumn("order_id", F.col("order_id").cast("integer"))
        .withColumn("order_status", F.trim("order_status"))
        .withColumn("status_timestamp", F.from_unixtime(F.col("status_timestamp").cast("long")).cast("timestamp"))
        .withColumn("year", F.year("status_timestamp"))
        .withColumn("month", F.month("status_timestamp"))
        .createOrReplaceTempView("order_batch_view")

    )

    sql_query = """
    MERGE INTO status_silver t
    USING order_batch_view s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.status_timestamp > t.status_timestamp
    THEN UPDATE SET *
    WHEN NOT MATCHED 
    THEN INSERT *
    """

    microbatch.sparkSession.sql(sql_query)

In [0]:
%sql
CREATE OR REPLACE TABLE status_silver
(
    order_id INT,
    order_status STRING,
    status_timestamp TIMESTAMP,
    year INT,
    month INT
)

In [0]:
(
    spark
.readStream
.table("spark_struct_retail.bronze.status_bronze")
.writeStream
.option("checkpointLocation", "/Volumes/spark_struct_retail/checkpoints/status_silver")
.foreachBatch(upsert_batch)
.trigger(availableNow = True)
.start()
)